In [1]:
# user-configurable variables

GENA_HOME = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch"
EXPERIMENT_CONFIG = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/downstream_tasks/expression_prediction/inference_example/inference.yaml"
CHECKPOINT_PATH = "/mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/full_model/pytorch_model.bin"

INFERENCE_DIR = "/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14"
JSON_DIR = "/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid812/json_14"
FORWARD_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.forward.csv"
REVERSE_INTERVALS_PATH = "/mnt/newdata/dpanc/benchmarking/data/human.valid.reverse.csv"

GENOME_PATH = "/mnt/newdata/dpanc/benchmarking/data/hg38.fna"
NUM_BEFORE = 512
TOKEN_LEN_FOR_FETCH = 15

DNA_TOKENIZER = None
TEXT_TOKENIZER = None
DNA_MAX_SEQ_LEN = None
TEXT_MAX_SEQ_LEN = None

PREDICTION_MATRIX_CSV = "valid812_updated_notebook_predictions.csv"
CORR_METHOD = "pearson"


In [2]:
import sys, os
import torch
import json
from pathlib import Path
from transformers import AutoTokenizer

# hydra imports; not really required if you will hard-code model params in future
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

# set GENALM_HOME environment variable to point to GENA_LM repo root; required to process config files
os.environ["GENALM_HOME"] = GENA_HOME 

# Keep torch/triton temporary files out of /tmp; /tmp can break compiled kernels here.
TASK_ROOT = "/mnt/newdata/dpanc/benchmarking/GENA_LM"
os.environ["TMPDIR"] = f"{TASK_ROOT}/cache/tmp"
os.environ["TRITON_CACHE_DIR"] = f"{TASK_ROOT}/cache/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{TASK_ROOT}/cache/torchinductor"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for _p in [os.environ["TMPDIR"], os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"], INFERENCE_DIR]:
    os.makedirs(_p, exist_ok=True)

import torch._dynamo
torch._dynamo.config.suppress_errors = True

sys.path.append(GENA_HOME)
sys.path.append(GENA_HOME+"/GENA_LM")

# import model
from downstream_tasks.expression_prediction.expression_model_final_sdpa import ExpressionCounts
from downstream_tasks.expression_prediction.expression_dataset_final import ExpressionDataset
from downstream_tasks.expression_prediction.inference_example.inference_input_utils import prepare_inference_inputs_from_intervals

/home/dpanc/benchmarking/GENA_LM/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# we have model parameters and other variables in config files; I made one for inference
experiment_config = EXPERIMENT_CONFIG

experiment_config_path = Path(experiment_config).expanduser().absolute()

with initialize_config_dir(str(experiment_config_path.parents[0])):
	experiment_config = compose(config_name=experiment_config_path.name)

model_kwargs = instantiate(experiment_config["model_kwargs"])

# initialize model
model = ExpressionCounts(**model_kwargs)

/tmp/ipykernel_3123114/842616341.py:6: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(experiment_config_path.parents[0])):
loading configuration file /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/modernbert_large/config.json
Model config ModernBertConfig {
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 50281,
  "classifier_activation": "silu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 1,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "embedding_dropout": 0.0,
  "eos_token_id": 50282,
  "global_attn_every_n_layers": 3,
  "global_rope_theta": 10000.0,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 1024,
  "initializer_cutoff_factor":

Using ModernGENA from /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/modernbert_large/
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []


loading configuration file config.json from cache at /home/dpanc/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25.self_attn.q_proj.weight
  - layers.25.self_attn.k_proj.weight
  - layers.25.self_attn.v_proj.weight
  - layers.25.self_attn.o_proj.weight
  - layers.25.self_attn.q_norm.weight
  - layers.25.self_attn.k_norm.weight
  - layers.25.mlp.gate_proj.weight
  - layers.25.mlp.up_proj.weight
  - layers.25.mlp.down_proj.weight
  - layers.25.input

In [4]:
# load checkpoint
checkpoint_path = CHECKPOINT_PATH 
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [5]:
# configure path-based inference inputs
# descriptions are loaded from all JSON files in json_dir,
# genes are taken from forward intervals (and optionally reverse intervals)

inference_dir = Path(INFERENCE_DIR) if INFERENCE_DIR is not None else Path(GENA_HOME) / "GENA_LM/downstream_tasks/expression_prediction/inference_example"
data_dir = inference_dir / "data"
json_dir = Path(JSON_DIR) if Path(JSON_DIR).is_absolute() else inference_dir / JSON_DIR
forward_intervals_path = Path(FORWARD_INTERVALS_PATH) if Path(FORWARD_INTERVALS_PATH).is_absolute() else inference_dir / FORWARD_INTERVALS_PATH
reverse_intervals_path = None if REVERSE_INTERVALS_PATH is None else (Path(REVERSE_INTERVALS_PATH) if Path(REVERSE_INTERVALS_PATH).is_absolute() else inference_dir / REVERSE_INTERVALS_PATH)
genome_path = Path(GENOME_PATH) 
num_before = int(NUM_BEFORE) 
token_len_for_fetch = TOKEN_LEN_FOR_FETCH


In [6]:
# helper that prepares descriptions from a folder with JSON files
# and tokenizes interval files exactly with the dataset logic

def prepare_inference_inputs(
	json_dir,
	forward_intervals_path,
	genome_path,
	reverse_intervals_path=None,
	gen_tokenizer=None,
	text_tokenizer_override=None,
	gen_max_seq_len_override=None,
	text_max_seq_len_override=None,
):
	gen_tokenizer_used = gen_tokenizer or dna_tokenizer
	text_tokenizer_used = text_tokenizer_override or text_tokenizer
	gen_max_seq_len_used = gen_max_seq_len_override or dna_max_seq_len
	text_max_seq_len_used = text_max_seq_len_override or text_max_seq_len

	return prepare_inference_inputs_from_intervals(
		json_dir=json_dir,
		forward_intervals_path=forward_intervals_path,
		reverse_intervals_path=reverse_intervals_path,
		genome_path=genome_path,
		gen_tokenizer=gen_tokenizer_used,
		text_tokenizer=text_tokenizer_used,
		gen_max_seq_len=gen_max_seq_len_used,
		text_max_seq_len=text_max_seq_len_used,
		cache_dir=inference_dir,
		num_before=num_before,
		token_len_for_fetch=token_len_for_fetch,
	)


In [7]:
# prepare tokenizers
dna_tokenizer_name = DNA_TOKENIZER or experiment_config["args_params"]["gen_tokenizer"]
text_tokenizer_name = TEXT_TOKENIZER or experiment_config["shared_dataset_params"]["text_tokenizer"]
dna_tokenizer = AutoTokenizer.from_pretrained(dna_tokenizer_name)
text_tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name, padding_side='left')

dna_max_seq_len = int(DNA_MAX_SEQ_LEN) if DNA_MAX_SEQ_LEN is not None else int(experiment_config["args_params"]["input_seq_len"])
text_max_seq_len = int(TEXT_MAX_SEQ_LEN) if TEXT_MAX_SEQ_LEN is not None else int(experiment_config["shared_dataset_params"]["text_max_seq_len"])

loading file tokenizer.json from cache at /home/dpanc/.cache/huggingface/hub/models--AIRI-Institute--gena-lm-bert-base-t2t/snapshots/4f1352bd4e820f1dba341047f54ad7795e083bf2/tokenizer.json
loading file tokenizer.model from cache at None
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /home/dpanc/.cache/huggingface/hub/models--AIRI-Institute--gena-lm-bert-base-t2t/snapshots/4f1352bd4e820f1dba341047f54ad7795e083bf2/special_tokens_map.json
loading file tokenizer_config.json from cache at /home/dpanc/.cache/huggingface/hub/models--AIRI-Institute--gena-lm-bert-base-t2t/snapshots/4f1352bd4e820f1dba341047f54ad7795e083bf2/tokenizer_config.json
loading file chat_template.jinja from cache at None
loading file vocab.json from cache at /home/dpanc/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/vocab.json
loading file merges.txt from cache at /home/dpanc/.cache/huggingface/hub/models--

In [8]:
# prepare interval-based inference inputs

prepared_inference = prepare_inference_inputs(
	json_dir=json_dir,
	forward_intervals_path=forward_intervals_path,
	genome_path=genome_path,
	reverse_intervals_path=reverse_intervals_path,
)

genes = prepared_inference["genes"]
experiments = prepared_inference["experiments"]
tokenized_DNA = prepared_inference["tokenized_DNA"]
tokenized_descriptions = prepared_inference["tokenized_descriptions"]

print("Gene token caches:", prepared_inference["gene_cache_paths"])
print("Description token cache:", prepared_inference["description_cache_path"])
print("Input IDs shape:", tokenized_DNA["input_ids"].shape)
tokenized_DNA


Gene token caches: {'forward': '/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14/inference_dataset_hash.forward.ed91c4ac73e84cf3.h5', 'reverse': '/mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14/inference_dataset_hash.reverse.75d764e3a7690768.h5'}
Description token cache: /mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14/json_14.9f8350900d93d327.Qwen_Qwen3-Embedding-0.6B.510.description.h5
Input IDs shape: torch.Size([3038, 1024])


{'input_ids': tensor([[    1,  1030,    31,  ...,     3,     3,     3],
         [    1,   246,   408,  ...,  2295,  4731,     2],
         [    1, 24971,  1525,  ..., 21461,   208,     2],
         ...,
         [    1,   151,  1622,  ...,    57,   946,     2],
         [    1,   570,    87,  ...,   161,   792,     2],
         [    1,   821,    48,  ...,   693,   240,     2]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]])}

In [9]:
# inspect tokenized experiment description

first_experiment = next(iter(experiments))
print("Experiments:", list(experiments.keys()))
print("Input IDs shape:", tokenized_descriptions[first_experiment]["input_ids"].shape)
tokenized_descriptions[first_experiment]


Experiments: ['ENCFF035CWS', 'ENCFF083EOC', 'ENCFF123KIW', 'ENCFF236XOK', 'ENCFF242BWW', 'ENCFF329ENM', 'ENCFF361XCF', 'ENCFF494KRC', 'ENCFF602HCV', 'ENCFF660EXG', 'ENCFF664WLU', 'ENCFF761SPP', 'ENCFF784MDF', 'ENCFF857JQM']
Input IDs shape: torch.Size([3038, 135])


{'input_ids': tensor([[   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         ...,
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643],
         [   395,    352,   4647,  ...,     23,     13, 151643]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]])}

In [ ]:
# cast inputs and run forward pass
# We run one gene at a time against all cell descriptions.
# This matches the benchmark-style shape and avoids putting all genes on GPU at once.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_ids_all = tokenized_DNA["input_ids"]
attention_mask_all = tokenized_DNA["attention_mask"]

gene_names = list(genes.keys())
cell_type_names = list(experiments.keys())

model = model.eval()
model.to(device)

outputs = {}
prediction_matrix_data = []

for gene_idx, gene_name in enumerate(gene_names):
    input_ids = input_ids_all[gene_idx:gene_idx + 1].to(device)
    attention_mask = attention_mask_all[gene_idx:gene_idx + 1].to(device)

    # desc_input_ids = torch.stack(
    #     [tokenized_descriptions[cell]["input_ids"][gene_idx] for cell in cell_type_names],
    #     dim=0,
    # ).unsqueeze(0).to(device)  # (B=1, N=cell types, D)

    # desc_attention_mask = torch.stack(
    #     [tokenized_descriptions[cell]["attention_mask"][gene_idx] for cell in cell_type_names],
    #     dim=0,
    # ).unsqueeze(0).to(device)  # (B=1, N=cell types, D)

    desc_ids_list = [tokenized_descriptions[cell]["input_ids"][gene_idx] for cell in cell_type_names]
    desc_mask_list = [tokenized_descriptions[cell]["attention_mask"][gene_idx] for cell in cell_type_names]
    max_desc_len = max(x.shape[0] for x in desc_ids_list)

    desc_input_ids = torch.stack(
        [torch.nn.functional.pad(x, (0, max_desc_len - x.shape[0]), value=tokenizer.pad_token_id) for x in desc_ids_list],
        dim=0,
    ).unsqueeze(0).to(device)

    desc_attention_mask = torch.stack(
        [torch.nn.functional.pad(x, (0, max_desc_len - x.shape[0]), value=0) for x in desc_mask_list],
        dim=0,
    ).unsqueeze(0).to(device)

    dataset_flag = torch.zeros(size=(1, len(cell_type_names)), device=device, dtype=torch.bool)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            desc_input_ids=desc_input_ids,
            desc_attention_mask=desc_attention_mask,
            dataset_flag=dataset_flag,
        )

    prediction_matrix_data.append(output["logits"][0, 0, :].detach().cpu().float().numpy())

    if (gene_idx + 1) % 100 == 0:
        print(f"Processed {gene_idx + 1}/{len(gene_names)} genes")


RuntimeError: stack expects each tensor to be equal size, but got [135] at entry 0 and [146] at entry 1

In [ ]:
# build prediction table and gene x cell-type matrix

import pandas as pd
import numpy as np

expression_matrix = pd.DataFrame(
    np.vstack(prediction_matrix_data),
    index=gene_names,
    columns=cell_type_names,
)

predictions_df = (
    expression_matrix
    .reset_index(names="Gene")
    .melt(id_vars="Gene", var_name="Cell Type", value_name="Predicted Expression")
)

display(predictions_df.head())
display(expression_matrix.head())


In [ ]:
# save gene x cell-type table to CSV

prediction_matrix_csv = Path(PREDICTION_MATRIX_CSV)
if not prediction_matrix_csv.is_absolute():
    prediction_matrix_csv = inference_dir / prediction_matrix_csv

expression_matrix.to_csv(prediction_matrix_csv)
print(f"Saved prediction matrix to: {prediction_matrix_csv}")
